# Example: AI Baseline and Validation Report

In this example, we implement the "zero-level AI baseline" (naive momentum rebalancing), compare all three strategies across the HMM-generated backtest scenario, and produce a formal validation report with pass/fail criteria for deployment readiness.

> **By the end of this example, you will be able to:**
> * Run the AI baseline strategy across all Monte Carlo paths
> * Compare three strategies head-to-head: AI engine, AI baseline, and buy-and-hold
> * Produce a validation report with explicit pass/fail criteria for production deployment

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

In [ ]:
let
    # Load data from Example 1 -
    bt_data = load(joinpath(_PATH_TO_DATA, "backtest-results.jld2"));
    sc_data = load(joinpath(_PATH_TO_DATA, "backtest-scenario.jld2"));

    global engine_bt = bt_data["engine"];
    global bh_bt = bt_data["buyhold"];
    global scenario = sc_data["scenario"];
    global B₀ = 10000.0;
    global offset = 84;
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];

    println("Loaded: $(scenario.n_paths) paths, engine + buy-and-hold results")
end

___
## Task 1: Run the AI Baseline Strategy
The "zero-level AI baseline" is a naive momentum strategy: rebalance monthly toward recent winners using a softmax over trailing 21-day returns. This mimics generic AI portfolio advice without any financial engineering.

> **What should you see?** The AI baseline will perform somewhere between buy-and-hold and the engine. Momentum works well in trending markets but gets whipsawed during regime shifts — exactly the kind of market the HMM produces.

In [ ]:
let
    println("Backtesting AI Baseline (momentum) across $(scenario.n_paths) paths...")
    global ai_bt = backtest_ai_baseline(scenario, tickers; B₀=B₀, offset=offset, lookback=21);

    println("\nAI Baseline Results:")
    println("  Median final wealth: \$$(round(median(ai_bt.final_wealth), digits=0))")
    println("  Median max drawdown: $(round(median(ai_bt.max_drawdowns) * 100, digits=1))%")
    println("  Median Sharpe: $(round(median(ai_bt.sharpe_ratios), digits=3))")
    println("  Failure rate: $(round(sum(ai_bt.final_wealth .< B₀) / scenario.n_paths * 100, digits=1))%")
end

___
## Task 2: Head-to-Head Strategy Comparison
We compare all three strategies side by side: final wealth distributions, drawdown distributions, and Sharpe ratio distributions. This reveals the relative strengths and weaknesses of each approach under regime-switching conditions.

> **What should you see?** Box plots showing the spread of outcomes for each strategy. The engine should show the best risk-adjusted profile (lower drawdown spread), while buy-and-hold may have the highest upside but also the worst tail losses.

In [ ]:
let
    # Comparison table -
    n = scenario.n_paths;
    comparison = DataFrame(
        "Metric" => ["Median Final Wealth (\$)", "10th Percentile Wealth (\$)",
            "90th Percentile Wealth (\$)", "Median Max Drawdown (%)",
            "Median Sharpe Ratio", "Failure Rate (%)"],
        "AI Engine" => [
            round(median(engine_bt.final_wealth), digits=0),
            round(quantile(engine_bt.final_wealth, 0.10), digits=0),
            round(quantile(engine_bt.final_wealth, 0.90), digits=0),
            round(median(engine_bt.max_drawdowns) * 100, digits=1),
            round(median(engine_bt.sharpe_ratios), digits=3),
            round(sum(engine_bt.final_wealth .< B₀) / n * 100, digits=1)
        ],
        "AI Baseline" => [
            round(median(ai_bt.final_wealth), digits=0),
            round(quantile(ai_bt.final_wealth, 0.10), digits=0),
            round(quantile(ai_bt.final_wealth, 0.90), digits=0),
            round(median(ai_bt.max_drawdowns) * 100, digits=1),
            round(median(ai_bt.sharpe_ratios), digits=3),
            round(sum(ai_bt.final_wealth .< B₀) / n * 100, digits=1)
        ],
        "Buy-and-Hold" => [
            round(median(bh_bt.final_wealth), digits=0),
            round(quantile(bh_bt.final_wealth, 0.10), digits=0),
            round(quantile(bh_bt.final_wealth, 0.90), digits=0),
            round(median(bh_bt.max_drawdowns) * 100, digits=1),
            round(median(bh_bt.sharpe_ratios), digits=3),
            round(sum(bh_bt.final_wealth .< B₀) / n * 100, digits=1)
        ]
    );

    println("═"^70)
    println("  THREE-WAY STRATEGY COMPARISON ($(n) HMM paths)")
    println("═"^70)
    pretty_table(comparison, tf=tf_markdown)
end

**Visualize:** Box plots comparing final wealth and max drawdown across all three strategies.

In [ ]:
let
    labels = ["AI Engine" "AI Baseline" "Buy-and-Hold"];

    p1 = boxplot(["AI Engine"], engine_bt.final_wealth, color=:steelblue, label="")
    boxplot!(p1, ["AI Baseline"], ai_bt.final_wealth, color=:green, label="")
    boxplot!(p1, ["Buy-and-Hold"], bh_bt.final_wealth, color=:coral, label="")
    hline!(p1, [B₀], linestyle=:dash, color=:black, label="Starting Capital", linewidth=1.5)
    ylabel!(p1, "Final Wealth (\$)")
    title!(p1, "Final Wealth Distribution")

    p2 = boxplot(["AI Engine"], engine_bt.max_drawdowns .* 100, color=:steelblue, label="")
    boxplot!(p2, ["AI Baseline"], ai_bt.max_drawdowns .* 100, color=:green, label="")
    boxplot!(p2, ["Buy-and-Hold"], bh_bt.max_drawdowns .* 100, color=:coral, label="")
    hline!(p2, [25.0], linestyle=:dash, color=:red, label="25% threshold", linewidth=1.5)
    ylabel!(p2, "Max Drawdown (%)")
    title!(p2, "Max Drawdown Distribution")

    plot(p1, p2, layout=(1, 2), size=(900, 450))
end

___
## Task 3: Formal Validation Report
We evaluate each strategy against the four deployment criteria: Sharpe ≥ 0.3, max drawdown ≤ 25%, failure rate ≤ 10%, and beats buy-and-hold. Strategies that pass all criteria are cleared for production deployment in Session 4.

> **What should you see?** A pass/fail report for each strategy. The AI engine is most likely to pass (designed with risk controls). The AI baseline may fail on drawdown (no risk controls). Buy-and-hold will likely fail on drawdown in HMM scenarios with crashes.

In [ ]:
let
    n = scenario.n_paths;
    bh_median_wealth = median(bh_bt.final_wealth);

    # Define validation criteria -
    criteria = Dict(
        "min_sharpe" => 0.3,
        "max_drawdown" => 25.0,
        "max_failure_rate" => 10.0,
        "min_wealth_vs_bh" => bh_median_wealth  # must beat buy-and-hold median
    );

    # Evaluate each strategy -
    strategies = [
        ("AI Engine (DD≤15%, τ≤50%)", engine_bt),
        ("AI Baseline (momentum)", ai_bt),
        ("Buy-and-Hold (equal-weight)", bh_bt)
    ];

    println("═"^70)
    println("  VALIDATION REPORT: Deployment Readiness")
    println("═"^70)
    println()

    for (label, bt) in strategies
        actuals = Dict(
            "min_sharpe" => median(bt.sharpe_ratios),
            "max_drawdown" => median(bt.max_drawdowns) * 100,
            "max_failure_rate" => sum(bt.final_wealth .< B₀) / n * 100,
            "min_wealth_vs_bh" => median(bt.final_wealth)
        );

        report = build(MyValidationReport;
            strategy_label=label, criteria=criteria, actuals=actuals);

        # Display -
        println("Strategy: $(label)")
        println("─"^50)

        report_df = DataFrame(
            "Criterion" => ["Sharpe ≥ 0.3", "Max DD ≤ 25%", "Failure Rate ≤ 10%", "Beats Buy-and-Hold"],
            "Threshold" => [0.3, 25.0, 10.0, round(bh_median_wealth, digits=0)],
            "Actual" => [
                round(actuals["min_sharpe"], digits=3),
                round(actuals["max_drawdown"], digits=1),
                round(actuals["max_failure_rate"], digits=1),
                round(actuals["min_wealth_vs_bh"], digits=0)
            ],
            "Result" => [
                report.passed["min_sharpe"] ? "PASS" : "FAIL",
                report.passed["max_drawdown"] ? "PASS" : "FAIL",
                report.passed["max_failure_rate"] ? "PASS" : "FAIL",
                report.passed["min_wealth_vs_bh"] ? "PASS" : "FAIL"
            ]
        );
        pretty_table(report_df, tf=tf_markdown)

        all_passed = all(values(report.passed));
        println("Overall: $(all_passed ? "PASSED — cleared for Session 4" : "FAILED — not deployment-ready")")
        println()
    end

    # Save validation results -
    save(joinpath(_PATH_TO_DATA, "validation-report.jld2"),
        Dict("engine" => engine_bt, "ai_baseline" => ai_bt, "buyhold" => bh_bt,
             "criteria" => criteria, "n_paths" => n));
end

___
## Summary
We completed the full validation pipeline for the AI rebalancing engine:

- **AI Baseline** provides a lower bound — if the engine can't beat naive momentum, the engineering isn't justified
- **Three-way comparison** across HMM-generated regime-switching paths reveals the true performance distribution
- **The validation report** provides explicit pass/fail gates for production deployment

Strategies that passed all four criteria are ready for Session 4, where we'll deploy the validated engine with live operating rules, sentiment monitoring, and human override protocols.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The strategies and validation criteria are pedagogical tools. Real-world deployment requires additional regulatory, legal, and operational validation.